In [14]:
import torch
from torch.autograd import Function
from torchvision import datasets,transforms
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.nn as nn
import torch.nn.functional as f
from pathlib import Path


In [15]:
train_data = datasets.MNIST(
    root='data',
    train = True,
    transform= transforms.ToTensor(),
    download=True,
)

test_data = datasets.MNIST(
    root='data',
    train=False,
    transform=transforms.ToTensor(),
)

In [16]:
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [17]:
#Hyperparameters
bach_size = 4
n_train = bach_size *125
N_TEST = bach_size *25
n_channel = 4
learning_rate  = 0.005


In [18]:
#define NeuralNetwork
class Net(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.conv = nn.Conv2d(1,1,4,stride=4)
        self.fc = nn.Linear(1*7*7,10)   
    
    def forward(self,x):
        # Propagate the input through the CNN layers
        x = self.conv(x)
        # Flatten the output from the convolutional layer
        x = torch.flatten(x,start_dim=1)
        x = f.relu(self.fc(x))
        
        return x
        
         

In [19]:
cnn = Net()
dataset = train_data
train_size = n_train
train_set , val_set = torch.utils.data.random_split(
    dataset,[train_size,len(dataset)- train_size]
)

In [20]:
print(len(train_set),len(val_set))

500 59500


In [25]:
train_loder = torch.utils.data.DataLoader(
    train_set,
    bach_size,
    shuffle=True
)

for data in train_loder:
    input, lable = data
    print(input.shape)
    print(lable.shape)
    
    output = cnn(input)
    
    print(output.shape)
    print(output)
    
    break

torch.Size([4, 1, 28, 28])
torch.Size([4])
torch.Size([4, 10])
tensor([[0.0538, 0.0000, 0.0000, 0.0000, 0.0369, 0.0000, 0.0000, 0.1055, 0.0000,
         0.2239],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0543, 0.0000, 0.1403, 0.0000,
         0.1733],
        [0.0133, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2145, 0.0000,
         0.1317],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2440, 0.0000,
         0.2243]], grad_fn=<ReluBackward0>)


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [11]:
#train 

import datetime
import os

dataset = train_data

cnn = Net()
cnn.to(device)

criterian = nn.CrossEntropyLoss()

#vi = 0.9vt + ndeltaL   teta = teta - vt
optimizer = optim.SGD(cnn.parameters(),learning_rate,momentum=0.90)
scheduler = lr_scheduler.StepLR(optimizer,step_size=1,gamma=0.8)


train_size = n_train
train_set, val_set = torch.utils.data.random_split(
    dataset,
    [train_size, len(dataset) - train_size]
)


train_loder = torch.utils.data.DataLoader(
    train_set,
    bach_size,
    shuffle=True
)
val_loader = torch.utils.data.DataLoader(
    val_set,
    batch_size=bach_size,
    shuffle=False
)

MODEL_PATH = Path('models')
MODEL_PATH.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "ImgClass-Quanvolv.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

RESUME_TRAINING = True

num_epochs = 45
loss_list = []
cnn.train()

if RESUME_TRAINING is False:
    print(f"Restore model state from {MODEL_SAVE_PATH}")

    if os.path.exists(MODEL_SAVE_PATH):
        model_dict = torch.load(MODEL_SAVE_PATH)
        inition_epoch = model_dict['epoch'] +1
        cnn.load_state_dict(model_dict['model_state_dict'])
        optimizer.load_state_dict(model_dict['optimizer_state_dict'])
        loss_list = model_dict['loss'].copy()
    
    else:
        print(f"No saved model state found. Training from scratch.")
        initial_epoch = 0
        loss_list = []
else:
    initial_epoch = 0
    loss_list = []


for epoch in range(num_epochs):
    ct = datetime.datetime.now()
    
    running_loss = []
    
    for i,data in enumerate(train_loder,0):
        inpute , lable = data
        inpute, lable = inpute.to(device), lable.to(device)
        optimizer.zero_grad()
        
        output = cnn(inpute)
        loss = criterian(output,lable)
        loss.backward()
        
        optimizer.step()
        running_loss.append(loss.item())
    
    scheduler.step()
    avg_loss = sum(running_loss) / len(running_loss)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.4f} - LR: {scheduler.get_last_lr()[0]}")
    
    torch.save({
        'epoch': epoch,
        'model_state_dict': cnn.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss_list,
    }, MODEL_SAVE_PATH) 
    print(f"Saving model state to {MODEL_SAVE_PATH}")    
        
    

Epoch [1/45] - Loss: 2.2763 - LR: 0.004
Saving model state to models\ImgClass-Quanvolv.pth
Epoch [2/45] - Loss: 1.6744 - LR: 0.0032
Saving model state to models\ImgClass-Quanvolv.pth
Epoch [3/45] - Loss: 1.0313 - LR: 0.00256
Saving model state to models\ImgClass-Quanvolv.pth
Epoch [4/45] - Loss: 0.8265 - LR: 0.0020480000000000003
Saving model state to models\ImgClass-Quanvolv.pth
Epoch [5/45] - Loss: 0.7656 - LR: 0.0016384000000000004
Saving model state to models\ImgClass-Quanvolv.pth
Epoch [6/45] - Loss: 0.7135 - LR: 0.0013107200000000005
Saving model state to models\ImgClass-Quanvolv.pth
Epoch [7/45] - Loss: 0.6660 - LR: 0.0010485760000000005
Saving model state to models\ImgClass-Quanvolv.pth
Epoch [8/45] - Loss: 0.6440 - LR: 0.0008388608000000005
Saving model state to models\ImgClass-Quanvolv.pth
Epoch [9/45] - Loss: 0.6275 - LR: 0.0006710886400000004
Saving model state to models\ImgClass-Quanvolv.pth
Epoch [10/45] - Loss: 0.6136 - LR: 0.0005368709120000003
Saving model state to mod

In [12]:
#accuracy

# Use a small subset of the full validation dataset
from torch.utils.data import SubsetRandomSampler

K = N_TEST # enter your length here
subsample_train_indices = torch.randperm(len(val_set))[:K]
val_loader = torch.utils.data.DataLoader(val_set, batch_size=bach_size, sampler=SubsetRandomSampler(subsample_train_indices))

correct = 0
total = 0
# Set the model to evaluation mode
cnn.eval()
with torch.inference_mode():
    for data in val_loader:
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)

        outputs = cnn(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f'Accuracy on the validation set: {100 * correct / total:.2f}%')

Accuracy on the validation set: 69.00%
